# 🧠 Pipeline NLP Analisis Sentimen & Pemetaan Narasi (Versi Colab GPU)

Notebook ini digunakan untuk mengolah data teks hasil *scraping* X (Twitter), melakukan **pembersihan noise**, **Sentiment Analysis (CardiffNLP RoBERTa GPU)**, **Pemetaan Narasi (Entman Framing)**, serta menghasilkan **grafik visualisasi profesional & laporan ringkasan untuk tim FISIP**.

### 📌 Cara Menggunakan:
1. Pastikan Runtime di Colab sudah diatur ke GPU: `Runtime > Change runtime type > T4 GPU`.
2. Jalankan **Sel 1** untuk menginstal pustaka NLP & Visualisasi.
3. Jalankan **Sel 2** untuk mengunggah file `asean_papua_x_tweets.csv` yang baru Anda dapatkan dari tahap sebelumnya.
4. Jalankan **Sel 3** (Pipeline Utama). Semua file CSV hasil analisis, laporan ringkasan FISIP, dan 4 grafik PNG akan **otomatis terunduh** ke komputer Anda!


In [1]:
!pip install transformers torch pandas matplotlib seaborn wordcloud nest_asyncio scikit-learn


In [2]:
# SEL 2: Upload File CSV Hasil Scraping
from google.colab import files
import os

print("Silakan unggah file 'asean_papua_x_tweets.csv':")
uploaded = files.upload()
for fn in uploaded.keys():
    print('File berhasil diunggah:', fn)


Silakan unggah file 'asean_papua_x_tweets.csv':


Saving asean_papua_x_tweets.csv to asean_papua_x_tweets.csv
File berhasil diunggah: asean_papua_x_tweets.csv


In [3]:
# ==============================================================================
# PIPELINE NLP ANALISIS SENTIMEN & PEMETAAN NARASI (VERSI GOOGLE COLAB GPU)
# ==============================================================================
# Petunjuk Penggunaan:
# 1. Buka https://colab.research.google.com
# 2. Pastikan Runtime Colab menggunakan GPU (Runtime > Change runtime type > T4 GPU)
# 3. Upload berkas process_nlp_colab.ipynb ATAU jalankan kode ini di Colab.
# ==============================================================================

# ------------------------------------------------------------------------------
# SEL 1: INSTALASI LIBRARIES (Jalankan sel ini terlebih dahulu)
# ------------------------------------------------------------------------------
"""
!pip install transformers torch pandas matplotlib seaborn wordcloud nest_asyncio scikit-learn
"""

# ------------------------------------------------------------------------------
# SEL 2 & 3: KODE UTAMA PIPELINE NLP (Salin seluruh kode di bawah ke dalam 1 Sel Colab)
# ------------------------------------------------------------------------------
import os
import re
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import torch
from transformers import pipeline

# Setting style grafik Matplotlib & Seaborn
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.family'] = 'sans-serif'

# ==============================================================================
# KONFIGURASI FILE & MODEL
# ==============================================================================
INPUT_CSV_FILENAME = "asean_papua_x_tweets.csv"
OUTPUT_CSV_FILENAME = "asean_papua_sentiment_analyzed.csv"
SUMMARY_TEXT_FILENAME = "ringkasan_analisis_fisip.txt"

# Model Sentimen SOTA Twitter dari CardiffNLP (Hugging Face)
SENTIMENT_MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# Kategori Narasi Entman Framing untuk Tim FISIP
NARRATIVE_CATEGORIES = {
    "HAM & UN Visit": [
        "human rights", "un", "commissioner", "rights", "abuse", "genocide",
        "violations", "investigation", "torture", "civilians", "hak asasi", "pbb"
    ],
    "Identitas Melanesia & Pasifik": [
        "melanesia", "melanesian", "msg", "pacific", "forum", "vanuatu",
        "solomon", "fiji", "kanaky", "solidarity", "brotherhood", "rumpun"
    ],
    "Kedaulatan & Autonomi": [
        "sovereignty", "autonomy", "referendum", "independence", "otsus",
        "freedom", "decolonisation", "ulmwp", "nkri", "kedaulatan", "otonomi"
    ],
    "Pembangunan & Stabilitas": [
        "development", "infrastructure", "conflict", "tni", "opm", "tpnpb",
        "military", "food", "economy", "pembangunan", "ekonomi", "keamanan"
    ]
}

# ==============================================================================
# FUNGSI PEMBERSIHAN TEKS (DATA CLEANSING)
# ==============================================================================
def clean_text_for_nlp(text: str) -> str:
    """
    Membersihkan URL, mention (@username), karakter aneh, dan spasi berlebih.
    """
    if not isinstance(text, str):
        return ""
    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Hapus mention @username
    text = re.sub(r'@\w+', '', text)
    # Hapus simbol khusus tapi pertahankan huruf dan kata
    text = re.sub(r'[^\w\s#]', ' ', text)
    # Hapus spasi berlebih dan newlines
    text = " ".join(text.split())
    return text

def filter_relevant_tweets(df: pd.DataFrame) -> pd.DataFrame:
    """
    Memfilter tuitan yang relevan dengan topik Papua & Melanesia (membuang spam/junk).
    """
    keywords = [
        "papua", "melanesia", "msg", "sovereignty", "human rights",
        "pacific", "ulmwp", "otsus", "indonesia", "vanuatu", "solomon"
    ]

    pattern = "|".join(keywords)

    # Filter berdasarkan full_text (case insensitive)
    is_relevant = df['full_text'].astype(str).str.contains(pattern, case=False, na=False)
    filtered_df = df[is_relevant].copy()
    print(f"[CLEANSING] Dari {len(df)} tuitan awal, {len(filtered_df)} tuitan relevan lolos filter.")
    return filtered_df

# ==============================================================================
# FUNGSI SENTIMENT ANALYSIS (TRANSFORMERS GPU)
# ==============================================================================
def run_sentiment_analysis(df: pd.DataFrame) -> pd.DataFrame:
    """
    Menjalankan Sentiment Analysis menggunakan Hugging Face Transformers di GPU.
    """
    device_id = 0 if torch.cuda.is_available() else -1
    device_name = f"GPU ({torch.cuda.get_device_name(0)})" if device_id == 0 else "CPU"
    print(f"\n[INFO] Menginisialisasi Model Sentimen '{SENTIMENT_MODEL_NAME}' pada {device_name}...")

    sentiment_pipe = pipeline(
        "sentiment-analysis",
        model=SENTIMENT_MODEL_NAME,
        tokenizer=SENTIMENT_MODEL_NAME,
        device=device_id,
        truncation=True,
        max_length=512
    )

    clean_texts = [clean_text_for_nlp(t) for t in df['full_text']]

    print("[INFO] Menjalankan inferensi sentimen...")
    results = sentiment_pipe(clean_texts)

    # Pemetaan Label CardiffNLP (positive, neutral, negative) -> (Positif, Netral, Negatif)
    label_mapping = {
        "positive": "Positif",
        "neutral": "Netral",
        "negative": "Negatif",
        "LABEL_0": "Negatif",
        "LABEL_1": "Netral",
        "LABEL_2": "Positif"
    }

    df['clean_text'] = clean_texts
    df['sentiment'] = [label_mapping.get(r['label'].lower(), r['label']) for r in results]
    df['sentiment_score'] = [round(r['score'], 4) for r in results]

    print("[SUCCESS] Sentiment Analysis selesai!")
    return df

# ==============================================================================
# FUNGSI PEMETAAN NARASI (ENTMAN FRAMING CLASSIFICATION)
# ==============================================================================
def classify_narrative(text: str) -> str:
    """
    Mengklasifikasikan tuitan ke dalam kategori narasi dominan berdasarkan keyword score.
    """
    text_lower = text.lower()
    scores = {}

    for category, kw_list in NARRATIVE_CATEGORIES.items():
        score = sum(1 for kw in kw_list if kw in text_lower)
        scores[category] = score

    best_category = max(scores, key=scores.get)
    if scores[best_category] == 0:
        return "Diplomasi & Wacana Umum"
    return best_category

def run_narrative_mapping(df: pd.DataFrame) -> pd.DataFrame:
    """
    Memetakan setiap tuitan ke dalam narasi dominan.
    """
    print("\n[INFO] Memetakan narasi dominan (Entman Framing)...")
    df['dominant_narrative'] = [classify_narrative(t) for t in df['clean_text']]
    print("[SUCCESS] Pemetaan narasi selesai!")
    return df

# ==============================================================================
# FUNGSI VISUALISASI & PEMBUATAN GRAFIK UNTUK FISIP
# ==============================================================================
def generate_visualizations(df: pd.DataFrame):
    """
    Membuat grafik visualisasi profesional untuk analisis FISIP.
    """
    print("\n[INFO] Membuat grafik visualisasi...")

    # 1. Grafik Pie Chart Distribusi Sentimen
    plt.figure(figsize=(7, 7))
    sentiment_counts = df['sentiment'].value_counts()
    colors = {'Positif': '#2ecc71', 'Netral': '#95a5a6', 'Negatif': '#e74c3c'}
    plot_colors = [colors.get(s, '#3498db') for s in sentiment_counts.index]

    plt.pie(
        sentiment_counts,
        labels=sentiment_counts.index,
        autopct='%1.1f%%',
        startangle=140,
        colors=plot_colors,
        explode=[0.05]*len(sentiment_counts),
        textprops={'fontsize': 12, 'weight': 'bold'}
    )
    plt.title("Distribusi Sentimen Wacana ASEAN (Isu Papua & Melanesia)", fontsize=14, pad=20, weight='bold')
    plt.tight_layout()
    plt.savefig("sentiment_distribution_pie.png", dpi=300)
    plt.close()
    print(" -> Grafik 'sentiment_distribution_pie.png' berhasil dibuat.")

    # 2. Grafik Bar Chart Distribusi Narasi Dominan
    plt.figure(figsize=(10, 5))
    narrative_counts = df['dominant_narrative'].value_counts()
    ax = sns.barplot(x=narrative_counts.values, y=narrative_counts.index, palette="viridis")
    plt.title("Distribusi Narasi Dominan (Entman Framing)", fontsize=14, weight='bold', pad=15)
    plt.xlabel("Jumlah Postingan", fontsize=12)
    plt.ylabel("Kategori Narasi", fontsize=12)

    # Tambahkan angka di atas bar
    for p in ax.patches:
        width = p.get_width()
        ax.annotate(f'{int(width)}', (width + 0.5, p.get_y() + p.get_height() / 2.),
                    ha='left', va='center', fontsize=10, color='black', weight='bold')

    plt.tight_layout()
    plt.savefig("narrative_distribution_bar.png", dpi=300)
    plt.close()
    print(" -> Grafik 'narrative_distribution_bar.png' berhasil dibuat.")

    # 3. Grafik Bar Chart Sentimen per Negara ASEAN
    if 'detected_country' in df.columns:
        plt.figure(figsize=(12, 6))
        country_sentiment = pd.crosstab(df['detected_country'], df['sentiment'])
        country_sentiment.plot(kind='bar', stacked=True, color=['#e74c3c', '#95a5a6', '#2ecc71'], figsize=(11, 6))
        plt.title("Distribusi Sentimen per Negara ASEAN", fontsize=14, weight='bold', pad=15)
        plt.xlabel("Negara", fontsize=12)
        plt.ylabel("Jumlah Postingan", fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.legend(title="Sentimen")
        plt.tight_layout()
        plt.savefig("sentiment_by_country_bar.png", dpi=300)
        plt.close()
        print(" -> Grafik 'sentiment_by_country_bar.png' berhasil dibuat.")

    # 4. WordCloud Teks
    all_text = " ".join(df['clean_text'])
    wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='tab10').generate(all_text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title("WordCloud Kata Paling Sering Muncul", fontsize=14, weight='bold', pad=15)
    plt.tight_layout()
    plt.savefig("wordcloud_asean_papua.png", dpi=300)
    plt.close()
    print(" -> Grafik 'wordcloud_asean_papua.png' berhasil dibuat.")

# ==============================================================================
# FUNGSI PEMBUATAN RINGKASAN TEKS UNTUK TEMAN FISIP
# ==============================================================================
def create_fisip_summary_report(df: pd.DataFrame):
    """
    Menyusun ringkasan eksekutif temuan data untuk diserahkan ke tim FISIP.
    """
    total = len(df)
    sent_counts = df['sentiment'].value_counts()
    narr_counts = df['dominant_narrative'].value_counts()

    pos_pct = round((sent_counts.get('Positif', 0) / total) * 100, 2)
    neu_pct = round((sent_counts.get('Netral', 0) / total) * 100, 2)
    neg_pct = round((sent_counts.get('Negatif', 0) / total) * 100, 2)

    summary_text = f"""
===============================================================================
           LAPORAN RINGKASAN EKSEKUTIF TEMUAN NLP (UNTUK TIM FISIP)
===============================================================================
Total Data Postingan Diindeks: {total} Tuitan

1. DISTRIBUSI SENTIMEN WACANA DIGITALS:
   - Positif : {sent_counts.get('Positif', 0)} tuitan ({pos_pct}%)
   - Netral  : {sent_counts.get('Netral', 0)} tuitan ({neu_pct}%)
   - Negatif : {sent_counts.get('Negatif', 0)} tuitan ({neg_pct}%)

2. PEMETAAN NARASI DOMINAN (ENTMAN FRAMING):
"""
    for narr, count in narr_counts.items():
        pct = round((count / total) * 100, 2)
        summary_text += f"   - {narr}: {count} tuitan ({pct}%)\n"

    summary_text += """
3. REKOMENDASI INTERPRETASI DIPLOMASI PUBLIK & TEORI:
   - Wacana bersentimen Negatif mayoritas didorong oleh isu HAM, tuntutan akses
     Komisioner PBB, serta seruan solidaritas Pasifik (MSG).
   - Narasi 'Identitas Melanesia & Pasifik' memegang peranan kunci dalam
     mengkonstruksi persepsi publik attentive luar negeri terhadap klaim diplomasi Indonesia.
===============================================================================
"""
    with open(SUMMARY_TEXT_FILENAME, "w", encoding="utf-8") as f:
        f.write(summary_text)

    print("\n" + summary_text)
    print(f"[SUCCESS] Laporan ringkasan FISIP disimpan di '{SUMMARY_TEXT_FILENAME}'.")

# ==============================================================================
# PIPELINE UTAMA EXEKUSI
# ==============================================================================
def main():
    print("="*70)
    print(" PIPELINE NLP ANALISIS SENTIMEN & TOPIC MODELING (VERSI COLAB GPU)")
    print("="*70)

    # 1. Cek keberadaan file CSV input
    if not os.path.exists(INPUT_CSV_FILENAME):
        print(f"\n[ERROR] File '{INPUT_CSV_FILENAME}' tidak ditemukan!")
        print("Silakan upload file CSV hasil scraping Anda ke Colab terlebih dahulu.")
        return

    # 2. Muat Data
    df = pd.read_csv(INPUT_CSV_FILENAME)
    print(f"[INFO] File CSV berhasil dimuat. Total baris: {len(df)}")

    # 3. Filtering & Cleansing
    df_clean = filter_relevant_tweets(df)

    if len(df_clean) == 0:
        print("[WARNING] Tidak ada tuitan relevan yang tersisa setelah filtering!")
        return

    # 4. Sentiment Analysis GPU
    df_analyzed = run_sentiment_analysis(df_clean)

    # 5. Narrative Mapping (Entman Framing)
    df_final = run_narrative_mapping(df_analyzed)

    # 6. Simpan File CSV Hasil Analisis
    df_final.to_csv(OUTPUT_CSV_FILENAME, index=False, encoding='utf-8-sig')
    print(f"\n[SUCCESS] Hasil analisis NLP disimpan di '{OUTPUT_CSV_FILENAME}'.")

    # 7. Buat Visualisasi Grafik
    generate_visualizations(df_final)

    # 8. Buat Laporan Ringkasan FISIP
    create_fisip_summary_report(df_final)

    # 9. Otomatis Download Seluruh File Hasil di Colab
    try:
        from google.colab import files
        print("\n[INFO] Mengunduh seluruh berkas hasil analisis ke komputer Anda...")
        files.download(OUTPUT_CSV_FILENAME)
        files.download(SUMMARY_TEXT_FILENAME)
        files.download("sentiment_distribution_pie.png")
        files.download("narrative_distribution_bar.png")
        if os.path.exists("sentiment_by_country_bar.png"):
            files.download("sentiment_by_country_bar.png")
        files.download("wordcloud_asean_papua.png")
    except Exception:
        pass

if __name__ == "__main__":
    main()



 PIPELINE NLP ANALISIS SENTIMEN & TOPIC MODELING (VERSI COLAB GPU)
[INFO] File CSV berhasil dimuat. Total baris: 427
[CLEANSING] Dari 427 tuitan awal, 411 tuitan relevan lolos filter.

[INFO] Menginisialisasi Model Sentimen 'cardiffnlp/twitter-roberta-base-sentiment-latest' pada GPU (Tesla T4)...


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (17 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<Figure size 1200x600 with 0 Axes>